# ASHRAE Human Baseline Notebook

This notebook records a true bank-id-authored, official-only human baseline for `ashrae-energy-prediction`. It intentionally keeps a conservative preprocessing spine and excludes leak-style corrections, holiday features, grouped interpolation, and public-kernel side inputs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
TRAIN_PATH = ROOT / "train.csv"
TEST_PATH = ROOT / "test.csv"
BUILDING_PATH = ROOT / "building_metadata.csv"
WEATHER_TRAIN_PATH = ROOT / "weather_train.csv"
WEATHER_TEST_PATH = ROOT / "weather_test.csv"


## Load Official Files

In [ ]:
train = pd.read_csv(TRAIN_PATH, nrows=200_000)
test = pd.read_csv(TEST_PATH, nrows=100_000)
building = pd.read_csv(BUILDING_PATH)
weather_train = pd.read_csv(WEATHER_TRAIN_PATH)
weather_test = pd.read_csv(WEATHER_TEST_PATH)

train.shape, test.shape, building.shape, weather_train.shape, weather_test.shape


## Join Building Metadata

In [ ]:
train = train.merge(building, on="building_id", how="left")
test = test.merge(building, on="building_id", how="left")

train[["building_id", "site_id", "primary_use", "square_feet"]].head()


## Join Split Weather Lookups

In [ ]:
train = train.merge(weather_train, on=["site_id", "timestamp"], how="left")
test = test.merge(weather_test, on=["site_id", "timestamp"], how="left")

weather_cols = [
    "air_temperature",
    "cloud_coverage",
    "dew_temperature",
    "precip_depth_1_hr",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
]
train[["site_id", "timestamp"] + weather_cols[:3]].head()


## Parse Timestamp

In [ ]:
train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"] = pd.to_datetime(test["timestamp"])

train["timestamp"].dtype, test["timestamp"].dtype


## Extract Timestamp Parts

In [ ]:
for frame in (train, test):
    frame["hour"] = frame["timestamp"].dt.hour
    frame["dayofweek"] = frame["timestamp"].dt.dayofweek
    frame["month"] = frame["timestamp"].dt.month

train[["timestamp", "hour", "dayofweek", "month"]].head()


## Label Encode Primary Use

In [ ]:
primary_use_vocab = {
    value: idx
    for idx, value in enumerate(sorted(train["primary_use"].dropna().unique()))
}

train["primary_use"] = train["primary_use"].map(primary_use_vocab).fillna(-1).astype(int)
test["primary_use"] = test["primary_use"].map(primary_use_vocab).fillna(-1).astype(int)

train[["building_id", "primary_use"]].head()


## Median Imputation

In [ ]:
impute_cols = [
    "year_built",
    "floor_count",
    "air_temperature",
    "cloud_coverage",
    "dew_temperature",
    "precip_depth_1_hr",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
]
train_medians = train[impute_cols].median()
train[impute_cols] = train[impute_cols].fillna(train_medians)
test[impute_cols] = test[impute_cols].fillna(train_medians)

train[impute_cols].isna().sum().sum(), test[impute_cols].isna().sum().sum()


## One-Hot Encode Meter

In [ ]:
train = pd.get_dummies(train, columns=["meter"], prefix="meter")
test = pd.get_dummies(test, columns=["meter"], prefix="meter")

meter_cols = sorted([col for col in train.columns if col.startswith("meter_")])
test = test.reindex(columns=train.columns.drop("meter_reading").tolist() + ["row_id"], fill_value=0)
meter_cols


## Log1p Square Feet

In [ ]:
train["square_feet"] = np.log1p(train["square_feet"])
test["square_feet"] = np.log1p(test["square_feet"])

train["square_feet"].head()


## Log1p Target

In [ ]:
target = np.log1p(train["meter_reading"])
target.head()


## Drop Raw Timestamp

In [ ]:
train = train.drop(columns=["timestamp"])
test = test.drop(columns=["timestamp"])

"timestamp" in train.columns, "timestamp" in test.columns


## Drop Test Row Id

In [ ]:
test_features = test.drop(columns=["row_id"])
train_features = train.drop(columns=["meter_reading"])

train_features.shape, test_features.shape


## Scope Boundary

In [ ]:
excluded_steps = [
    "site-0 leak correction",
    "holiday features",
    "public-kernel side inputs",
    "grouped interpolation of weather gaps",
    "heavy lag or rolling feature branches",
]
excluded_steps
